In [1]:
import pandas as pd
import math
import numpy as np



In [2]:
hospital_data_s2 = pd.read_csv("../data/curated/hospital_data/hospital_data_curated_sa2.csv")
rental_data_distance = pd.read_csv("../data/curated/rental_data_distance_sf.csv")
rental_data_distance["coordinates"] = rental_data_distance["coordinates"].apply(eval)
hospital_data_s2["coordinate"] = hospital_data_s2["coordinate"].apply(eval)
hospital_data_s2

,formal name,emergency capable,location address,suburb,postcode,category,agency type,coordinate,sa2_code_2021,sa2_name_2021
0,albert road clinic,no,31-33 albert road,south melbourne,3205,private,private hospital,"(-37.8354573, 144.9705465)",206051512.0,South Melbourne
1,albert road clinic,no,31-33 albert road,south melbourne,3205,private,private hospital,"(-37.8354573, 144.9705465)",206051511.0,Port Melbourne Industrial
2,albury wodonga health,yes,69 vermont street,wodonga,3690,public,public hospital,"(-36.1320403, 146.8800037)",204031491.0,Baranduda - Leneva
3,albury wodonga health,yes,69 vermont street,wodonga,3690,public,public hospital,"(-36.1320403, 146.8800037)",204031073.0,West Wodonga
4,albury wodonga health,yes,69 vermont street,wodonga,3690,public,public hospital,"(-36.1320403, 146.8800037)",204031492.0,Wodonga
...,...,...,...,...,...,...,...,...,...,...
492,yarram and district health service,no,85 commercial road,yarram,3971,public,public hospital,"(-38.557082550000004, 146.67819733634434)",205051104.0,Yarram
493,yarram and district health service,no,85 commercial road,yarram,3971,public,public hospital,"(-38.557082550000004, 146.67819733634434)",205041094.0,Churchill
494,yarrawonga health,no,33 piper street,yarrawonga,3730,public,public hospital,"(-36.0110552, 146.0059269)",216021414.0,Yarrawonga
495,yarrawonga health,no,33 piper street,yarrawonga,3730,public,public hospital,"(-36.0110552, 146.0059269)",216021412.0,Moira


In [3]:
EARTH_RADIUS = 6371
def calculate_distance(coordinate1, coordinate2):
    """ Function calculates the haversine distance between two coordinates """
    # Convert latitude and longitude from degrees to radians
    lat1 = np.radians(coordinate1[0])
    lon1 = np.radians(coordinate1[1])
    lat2 = np.radians(coordinate2[0])
    lon2 = np.radians(coordinate2[1])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = EARTH_RADIUS * c
    
    return distance
    

In [4]:
def distance_closest_hospital(rental_data_coordinate, rental_data_sa2, new_col_names, hospital_data):
    """ Function finds the distance to the closest hospital from the rental property """

    distance_dict = {}

    # Find closest university by comparing with all hospitals in victoria
    for index, hospital in hospital_data.iterrows():
        distance_dict[hospital["formal name"]] = calculate_distance(rental_data_coordinate, hospital["coordinate"])
    if distance_dict:
        # return closest hopsital
        return pd.Series({new_col_names[0]: min(distance_dict.items(), key=lambda x: x[1])[0],
                        new_col_names[1]: min(distance_dict.items(), key=lambda x: x[1])[1]})

In [5]:
new_col_names = ["closest hospital", "distance to closest hospital"]
new_col_hospital = rental_data_distance.apply(lambda x: distance_closest_hospital(
    x["coordinates"], x["SA2_CODE21"], new_col_names, hospital_data_s2), axis=1)
rental_data_distance = pd.concat([rental_data_distance, new_col_hospital], axis=1)

In [6]:
rental_data_distance.to_csv("../data/curated/rental_data_distance_sf.csv", index=False)

In [7]:
rental_data = pd.read_csv("../data/curated/rental_data_sf.csv")
rental_data_distance = pd.read_csv("../data/curated/rental_data_distance_sf.csv")
print(len(rental_data_distance))
print(len(rental_data))

11409
11409


In [8]:
rental_data_distance

,name,cost_text,coordinates,property type,beds,baths,parking,title,description,property features,...,SA2_CODE21,SA2_NAME21,closest primary school,distance primary school,closest secondary school,distance secondary school,closest university campus,distance university campus,closest hospital,distance to closest hospital
0,Rooms/227 Elgar Road Surrey Hills VIC 3127,$120.00,"[-37.8353315, 145.1114189]",House,1,2,6,Single room in rooming house - CURRENTLY NO RO...,"['** BOOK INSPECTION ONLINE **', '*Currently n...","['Furnished*', 'Broadband internet access*', '...",...,207031167.0,Surrey Hills (East) - Mont Albert,our lady of perpetual succour,0.586504,our lady of sion college,2.491226,monash university-notting hill,7.430317,box hill hospital,2.104719
1,30 Daly Boulevard Highton VIC 3216,$130,"[-38.1962932, 144.3060777]",House,9,2,2,Student Accommodation - 1 Room Available!,['We have 1 Rooms Available to Rent.Room 2 - $...,"['Gas*', 'Internal Laundry*', 'Secure Parking*...",...,203021042.0,Highton,christian college geelong,1.372612,grovedale college,2.438191,deakin university-waurn ponds,0.675843,"st john of god hospital, geelong",6.688022
2,Berth 5274 Martha Cove Waterway Safety Beach V...,$140 per week,"[-38.3138012, 145.0045603]",House,1,1,0,Boat Berth B5274,['BOAT BERTH AVAILABLE-B5274-Located in Martha...,[],...,214021377.0,Dromana,dromana primary,4.858273,dromana secondary college,3.301447,monash university-notting hill,47.278807,the bays hospital,10.766412
3,18 Cedarville Close Highton VIC 3216,$140,"[-38.19508, 144.3154857]",House,1,2,6,Room 4 rent!,['Rooms for rent in this student share house. ...,"['Built in wardrobes', 'Internal Laundry', 'Ga...",...,203021042.0,Highton,christian college geelong,1.114181,grovedale college,1.998214,deakin university-waurn ponds,1.502495,"st john of god hospital, geelong",6.057051
4,123 Thornhill Road Highton VIC 3216,$150,"[-38.1845196, 144.3107906]",House,1,4,3,Room for rent near Uni,['Rooms within walking distance to Waurn Ponds...,"['Ensuite', 'Furnished', 'Built in wardrobes*'...",...,203021042.0,Highton,christian college geelong,0.215009,belmont high,1.989564,deakin university-waurn ponds,1.862509,"st john of god hospital, geelong",5.472987
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11404,Enquire For Address Brighton VIC 3186,"$4,400.00","[-37.9044002, 144.9997396]",House,4,4,2,Golden Mile Luxury,"['Inspect by Appointment Only ', 'This family ...","['Study', 'Swimming Pool', 'Ducted Cooling', '...",...,208011169.0,Brighton (Vic.),firbank grammar,0.311431,star of the sea college,0.891118,monash university-notting hill,11.507926,epworth rehabilitation brighton,0.931372
11405,335 The Esplanade Indented Head VIC 3223,"$4,700 pw /$700 pn High, $3,775 pw/$545 pn Low","[-38.1439923, 144.7147899]",House,4,3,0,"335 The Esplanade, Indented Head",['This stylish 4 bedroom home will surely make...,"['Ensuite', 'Deck', 'Dishwasher', 'Ducted Heat...",...,203031051.0,Portarlington,st leonards primary,3.095969,bellarine secondary college,14.695112,the university of melbourne-parkville,44.215772,the geelong clinic,28.018349
11406,8701/7 Riverside Quay Southbank VIC 3006,"$5,000.00","[-37.8213342, 144.9646982]",Apartment / Unit / Flat,3,3,4,On Top of the World,"[""A brilliant opportunity to lease the exclusi...",[],...,206041508.0,Southbank (West) - South Wharf,south melbourne primary,1.327110,victorian college of the arts secondary,0.766691,the university of melbourne-southbank,0.615756,albert road clinic,1.652282
11407,1 Weston Street Balwyn VIC 3103,"$9,950 weekly","[-37.8117577, 145.0775507]",House,6,7,8,"Lifestyle, Modern & Luxury Living Awaits",['**Scheduling your property inspection is now...,"['Air conditioning*', 'Built in wardrobes*', '...",...,207011147.0,Balwyn,balwyn primary,0.414506,canterbury girls secondary college,1.204451,the university of melbourne-parkville,10.386190,mercy health - o'connell family centre,1.204835
